# Pareto-фронт

Визуализация Парето-оптимальных кандидатов после фильтров — для UI «карта компромиссов» (TCO vs price, TCO vs reliability и др.).

Идея: TOPSIS даёт **один линейный порядок**, но рынок многомерный. Парето-фронт — **множество** недоминируемых кандидатов: для каждого из них нет другой машины, которая **одновременно** лучше по всем критериям. UI может показать пользователю карту «дешевле / лучше TCO / надёжнее» и подсветить машины-«углы фронта».

Алгоритм: для кандидата A и B говорим, что A **доминирует** B, если по каждому критерию A не хуже B и хотя бы по одному строго лучше. Парето-фронт — множество недоминируемых. Multi-layer NSGA-style: «снять» первый фронт и повторить.

Реализация: `ml/recommendation/pareto.py` (numpy, O(n²·m), m = число критериев; для 875 машин и 6 критериев работает за ~ 100 мс).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT.parent))

from ml.recommendation import (
    UserProfile, TopsisStrategy, RecommendationService,
    pareto_front, pareto_front_2d, compute_pareto_layers,
)
from ml.recommendation.tco_calc import TCOCalculator, load_seed
from ml.recommendation.filters import build_filters_from_profile
from ml.recommendation.criteria import DEFAULT_CRITERIA

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Imports OK')

## 1. Загрузка каталога и подготовка кандидатов

Используем сценарий «семья + дача» (case 3): кроссовер J_CROSS, бюджет 3.5 М, регион Краснодарский край.

In [ ]:
seed = load_seed()
calc = TCOCalculator(seed)
service = RecommendationService(tco_calc=calc, strategy=TopsisStrategy())

profile = UserProfile(
    budget_rub=3_500_000,
    region_id=33,
    mileage_per_year_km=18000,
    weight_preset='balanced',
    preferred_segments=['J_CROSS'],
    allowed_body_types=['crossover', 'suv'],
    min_seats=5, min_power_hp=120, min_clearance_mm=170, min_cargo_volume_l=400,
)

augmented = service._augment_catalog(seed['modifications'])
filters = build_filters_from_profile(profile)
filtered = service._apply_filters(augmented, filters)
with_crit = service._compute_dynamic_criteria(filtered, profile)

print(f'Catalog: {len(augmented)} -> after filters: {len(filtered)} candidates')
print(f'Criteria columns: {[c.name for c in DEFAULT_CRITERIA]}')
with_crit[['make_name','model_name','tco_5y','purchase_price','reliability_score','power_hp','cargo_volume_l','depreciation_5y_pct']].head(5)

## 2. 6-мерный Парето-фронт

Размер фронта — нижняя оценка «sweet-spots»: ниже него любой выбор хуже хотя бы одного из них по всем критериям одновременно.

In [ ]:
front_idx = pareto_front(with_crit, DEFAULT_CRITERIA)
print(f'6D Pareto front: {len(front_idx)} candidates ({len(front_idx)/len(with_crit)*100:.1f}% of filtered)')

with_layers = compute_pareto_layers(with_crit, max_layers=3)
layer_counts = with_layers['pareto_layer'].value_counts().sort_index()
print(f'Pareto layers: {layer_counts.to_dict()}')
print('  layer 0  = глубже 3-го фронта (можно скрывать в UI)')
print('  layer 1  = первый Pareto-фронт (лучшие)')
print('  layer 2  = второй фронт после удаления первого')
print('  layer 3  = третий фронт')

## 3. 2D-карта: TCO vs Покупная цена

Главная плоскость для UI: пользователь сравнивает «дешевле сразу» vs «дешевле в эксплуатации». Зелёные точки — Парето-оптимальные.

In [ ]:
front_2d = pareto_front_2d(with_crit, 'tco_5y', 'minimize', 'purchase_price', 'minimize')
print(f'2D Pareto front (TCO vs Price): {len(front_2d)} candidates')

fig, ax = plt.subplots(figsize=(11, 7))
is_front = np.zeros(len(with_crit), dtype=bool)
is_front[front_2d] = True

ax.scatter(with_crit.loc[~is_front, 'purchase_price']/1e6,
           with_crit.loc[~is_front, 'tco_5y']/1e6,
           c='lightgray', s=30, alpha=0.6, label='Dominated')
ax.scatter(with_crit.loc[is_front, 'purchase_price']/1e6,
           with_crit.loc[is_front, 'tco_5y']/1e6,
           c='green', s=80, edgecolor='darkgreen', label=f'Pareto-optimal ({len(front_2d)})')

front_sorted = with_crit.iloc[front_2d].sort_values('purchase_price')
ax.plot(front_sorted['purchase_price']/1e6, front_sorted['tco_5y']/1e6,
        c='green', alpha=0.4, linewidth=1, linestyle='--')

for _, row in front_sorted.iterrows():
    ax.annotate(f"{row['make_name']}\n{row['model_name']}",
                (row['purchase_price']/1e6, row['tco_5y']/1e6),
                fontsize=7, alpha=0.85, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Purchase price, M RUB')
ax.set_ylabel('TCO 5y, M RUB')
ax.set_title('Case 3: семья + дача J_CROSS — Pareto front (TCO 5y vs price)\n(both axes minimize: bottom-left corner is best)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 4. 2D-карта: TCO vs Надёжность

Вторая ключевая плоскость: «дешевле в эксплуатации» vs «надёжнее». Pareto-фронт здесь рисуется в **левом верхнем углу** (TCO ↓, reliability ↑).

In [ ]:
front_rel = pareto_front_2d(with_crit, 'tco_5y', 'minimize', 'reliability_score', 'maximize')
print(f'2D Pareto front (TCO vs Reliability): {len(front_rel)} candidates')

fig, ax = plt.subplots(figsize=(11, 7))
is_front = np.zeros(len(with_crit), dtype=bool)
is_front[front_rel] = True

ax.scatter(with_crit.loc[~is_front, 'reliability_score'],
           with_crit.loc[~is_front, 'tco_5y']/1e6,
           c='lightgray', s=30, alpha=0.6, label='Dominated')
ax.scatter(with_crit.loc[is_front, 'reliability_score'],
           with_crit.loc[is_front, 'tco_5y']/1e6,
           c='steelblue', s=80, edgecolor='navy', label=f'Pareto-optimal ({len(front_rel)})')

front_sorted = with_crit.iloc[front_rel].sort_values('reliability_score')
ax.plot(front_sorted['reliability_score'], front_sorted['tco_5y']/1e6,
        c='steelblue', alpha=0.4, linewidth=1, linestyle='--')

for _, row in front_sorted.iterrows():
    ax.annotate(f"{row['make_name']}\n{row['model_name']}",
                (row['reliability_score'], row['tco_5y']/1e6),
                fontsize=7, alpha=0.85, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Reliability score (0-1, higher = better)')
ax.set_ylabel('TCO 5y, M RUB (lower = better)')
ax.set_title('Case 3 — Pareto front (TCO 5y vs reliability)\n(top-left corner is best)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 5. Multi-layer Pareto + цвет по слоям

NSGA-стиль: первый фронт удалили, на остатке нашли второй фронт, и так далее. Это даёт **градации качества**: первый слой — «idealлагли», второй — «почти идеалы», третий — «хорошие».

In [ ]:
with_layers_5 = compute_pareto_layers(with_crit, max_layers=5)
fig, ax = plt.subplots(figsize=(11, 7))

colors = {0: 'lightgray', 1: 'darkgreen', 2: 'limegreen', 3: 'gold', 4: 'orange', 5: 'tomato'}
labels = {0: 'layer >5', 1: 'layer 1 (best)', 2: 'layer 2', 3: 'layer 3', 4: 'layer 4', 5: 'layer 5'}
for layer in sorted(with_layers_5['pareto_layer'].unique()):
    sub = with_layers_5[with_layers_5['pareto_layer'] == layer]
    ax.scatter(sub['purchase_price']/1e6, sub['tco_5y']/1e6,
               c=colors.get(layer, 'gray'), s=40 if layer == 0 else 70,
               edgecolor='black' if layer > 0 else 'none', linewidth=0.5,
               alpha=0.5 if layer == 0 else 0.9, label=f'{labels.get(layer, layer)} (n={len(sub)})')

ax.set_xlabel('Purchase price, M RUB')
ax.set_ylabel('TCO 5y, M RUB')
ax.set_title('Case 3 — Multi-layer Pareto (NSGA-style, max 5 layers)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print('\n  Layer distribution:')
print(with_layers_5['pareto_layer'].value_counts().sort_index().to_string())

## 6. Pareto vs TOPSIS top-10 — пересечение

**Гипотеза:** все TOPSIS-победители обязаны быть Парето-оптимальны. Это математически обязано выполняться: если кандидат A доминируется кандидатом B, то у B норм. координаты во всех критериях не хуже A → его relative closeness ≥ A → TOPSIS не поставит A выше B.

Проверяем эмпирически (это unit-test для корректности TOPSIS-реализации).

In [ ]:
result = service.recommend(profile, top_k=10)
topsis_top10_modids = [it.modification_id for it in result.top]
print(f'TOPSIS top-10:')
for it in result.top:
    print(f'  #{it.rank}  {it.make_name:15} {it.model_name:25} score={it.score:.3f}')

front_full = pareto_front(with_crit, DEFAULT_CRITERIA)
front_modids = set(with_crit.iloc[front_full]['modification_id'].values)

outside_pareto = [m for m in topsis_top10_modids if m not in front_modids]
print(f'\n  TOPSIS top-10 ∩ Pareto-front: {10 - len(outside_pareto)}/10')
if outside_pareto:
    print(f'  ⚠️  vне Pareto-фронта: {outside_pareto}')
else:
    print('  ✅  все TOPSIS-победители — Pareto-оптимальны (как и ожидалось теоретически)')

## 7. Итоги

- **Pareto-фронт** даёт пользователю **дополнение** к TOPSIS-топу: видно «всю карту компромиссов», а не только один линейный порядок.
- **2D-проекции** (`pareto_front_2d`) — самые ценные для UI: пользователь интерактивно меняет оси (цена / TCO / надёжность / мощность) и сразу видит «sweet-spots».
- **Multi-layer (NSGA)** позволяет делать градации качества: first layer = «идеалы», layers 2-5 = «почти-идеалы». Дальше скрыть.
- TOPSIS top-10 ⊆ Pareto-фронт — sanity-check корректности.